# OmniVoice — TTS & Voice Cloning

The neural TTS/cloning model (~2.5 GB). Fast on GPU, slow on CPU. This is the real model, not the preview.

**Model:** `k2-fsa/OmniVoice`  |  **Feature:** Text to speech & voice cloning

Runs on CPU or GPU. Cells: setup → data → load → test → benchmark → accuracy → training notes → copyright.


## 1. Setup - code + dependencies


In [ ]:
import os, sys, subprocess, time, io, urllib.request, platform
from pathlib import Path
REPO = 'https://github.com/Zubair-hussain/Xovato-cpu-ai-studio.git'
if Path('app').exists(): ROOT = Path('.').resolve()
elif Path('../app').exists(): ROOT = Path('..').resolve()
else:
    subprocess.run(['git','clone','--depth','1',REPO], check=True)
    ROOT = Path('Xovato-cpu-ai-studio').resolve()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','pillow','numpy','onnxruntime','soundfile'], check=False)
print('root:', ROOT, '| python', platform.python_version())

## 2. Data (test set)


In [ ]:
TEXT = 'This voice is generated locally by the OmniVoice model.'
REF_AUDIO = ''  # optional: path to a 10-30s clean clip for cloning
REF_TEXT  = ''  # the exact transcript of that clip
print('text set. cloning:', bool(REF_AUDIO))

## 3. Download & load the model


In [ ]:
import torch
subprocess.run([sys.executable,'-m','pip','install','-q','omnivoice','huggingface_hub'], check=False)
from huggingface_hub import snapshot_download
target = ROOT/'models'/'omnivoice'/'k2-fsa_OmniVoice'
snapshot_download('k2-fsa/OmniVoice', local_dir=str(target),
                  allow_patterns=['*.json','*.safetensors','*.txt','*.model','audio_tokenizer/*'])
os.environ['ENHANCEAI_USE_REAL_OMNIVOICE']='true'
os.environ['ENHANCEAI_OMNIVOICE_QUANTIZE_INT8']= 'false' if torch.cuda.is_available() else 'true'
from app.omnivoice_runtime import model_ready; print('model ready:', model_ready(target))

## 4. Inference test


In [ ]:
from app.config import get_settings
from app.voice_engine import voice_engine
from app.voice_styles import resolve_style
import soundfile as sf
style = resolve_style('calm')
kw = dict(text=TEXT, style=style)
if REF_AUDIO: kw.update(ref_audio=Path(REF_AUDIO), ref_text=REF_TEXT, cache_key='demo')
r = voice_engine.synthesize(get_settings(), **kw)
sf.write('omnivoice.wav', r.audio, r.sample_rate)
print('cloned:', r.cloned, '| gen_s:', round(r.generate_seconds,2))
from IPython.display import Audio; Audio('omnivoice.wav')

## 5. Benchmark (latency)


In [ ]:
import time; from app.config import get_settings
t=time.perf_counter(); r=voice_engine.synthesize(get_settings(), text=TEXT, style=resolve_style('neutral'))
dur=len(r.audio)/r.sample_rate
print(f'{r.generate_seconds:.2f}s to generate {dur:.1f}s audio -> RTF {r.generate_seconds/max(dur,1e-6):.2f}x')

## 6. Accuracy signal


In [ ]:
print('Quality scales with num_step and guidance_scale. Objective checks:')
print(' - speaker similarity: cosine of speaker embeddings (resemblyzer) ref vs output')
print(' - intelligibility: WER of an ASR transcript of the output vs the input text')
print('See the OmniVoice model card for reported zero-shot metrics.')

## 7. Training / fine-tuning notes

OmniVoice is a large pretrained model; full training needs a substantial GPU cluster and the k2-fsa training code. The practical 'training' here is **zero-shot voice cloning** - give a reference clip + its transcript and the model adapts with no gradient steps.


## 8. Copyright


In [ ]:
# ====================================================================
# COPYRIGHT NOTICE - DO NOT REMOVE
# ====================================================================
# (c) Zubair Hussain - Xovato AI Studio.
# These notebooks, the benchmark/accuracy harness, and the surrounding
# tooling are the work of Zubair Hussain. You may use and modify them for
# your own projects; keep this attribution intact, including in the
# background-removal notebook and any derived outputs.
print('(c) Zubair Hussain - Xovato AI Studio')